# PVGIS to temperature and power time series Converter

Convert a PVGIS file in the example directory (```pv_timeseries_pvgis.csv```) to two csv files in the example directory:

| Filename                     |        Corresponding blocks        | Description                                                                                                               |
|------------------------------|----------------------------------|---------------------------------------------------------------------------------------------------------------------------|
| ```pv_timeseries_file.csv``` |  ```PVSource```, ```WindSource```  | 15 min time series with specific power (```power_spec```), wind speed (```speed_wind```) and temperature (```temp_air```) |
| ```temperature_air.csv```    |           ```Scenario```           | 15 min time series with temperature (```temp_air```)                                                                      |


In [ ]:
from pathlib import Path

import pandas as pd

#### Set variables

In [ ]:
PATH_EXAMPLE = Path.cwd().resolve().parent / "revoletion" / "example"
FILE_INPUT = PATH_EXAMPLE / "pv_timeseries_pvgis.csv"

# Set target timezone for PVGIS data. The data is in UTC and needs to be converted to the local timezone.
TIMEZONE = "Europe/Berlin"

#### Read PVGIS data and convert time information to local timezone.


In [ ]:
df = pd.read_csv(
    FILE_INPUT,
    engine="python",  # required for skipfooter
    skiprows=10,
    skipfooter=13,
)

# convert time
df["time"] = (
    pd.to_datetime(df["time"], format="%Y%m%d:%H%M").dt.round("h").dt.tz_localize("UTC").dt.tz_convert(TIMEZONE)
)

# set time as index
df.set_index("time", drop=True, inplace=True)

 #### Resample the data to 15 min intervals

In [ ]:
# already ends at 01:00 due to timezone shift -> add another hour to avoid data losses
df = df.reindex(df.index.union(df.index.shift(periods=1, freq="15min")[-1:]))
df = df.resample("15min").ffill().bfill()
df = df.iloc[:-1, :]

#### Extract relevant columns and save as CSV

In [ ]:
df = df.rename(columns={"P": "power_spec", "WS10m": "speed_wind", "T2m": "temp_air"})
df[["power_spec", "speed_wind", "temp_air"]].to_csv((PATH_EXAMPLE / "pv_timeseries_file.csv"), index=True)
df[["temp_air"]].to_csv((PATH_EXAMPLE / "temperature_air.csv"), index=True)